In [ ]:
%pip install -q pymupdf python-stdnum openpyxl pillow transformers tqdm

In [ ]:
import os, re, json, hashlib, time, gc
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import fitz
import torch
from PIL import Image
from stdnum import iban as iban_lib
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from transformers import AutoProcessor, AutoModelForVision2Seq
from tqdm.notebook import tqdm

print("✅ Imports OK")

In [ ]:
# ── Chemins ──────────────────────────────────────────────────────────────────
MODEL_PATH       = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main"
INPUT_DIR        = Path("/mnt/data/transferts_in")
OUTPUT_DIR       = Path("/mnt/data/transferts_out")
JSON_ARCHIVE_DIR = OUTPUT_DIR / "json_archive"
EXCEL_OUTPUT     = OUTPUT_DIR / "audit_transferts.xlsx"
CHECKPOINT_FILE  = OUTPUT_DIR / "checkpoint.json"
OV_VIERGE_PATH   = OUTPUT_DIR / "ov_vierge.jpeg"

for d in [INPUT_DIR, OUTPUT_DIR, JSON_ARCHIVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── GPU & modèle ──────────────────────────────────────────────────────────────
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
GPU_BATCH_SIZE  = 4      # pages en parallèle (hors OV vision)
MAX_NEW_TOKENS  = 400    # suffisant pour tous les JSON
IMAGE_MAX_SIZE  = 1120   # pixels côté long max
MIN_TEXT_CHARS  = 80     # seuil texte natif vs vision

# ── Excel ─────────────────────────────────────────────────────────────────────
EXCEL_FLUSH_EVERY = 50

# ── Extensions supportées ─────────────────────────────────────────────────────
SUPPORTED_EXT = {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

# ── Couleurs Excel (format ARGB openpyxl) ────────────────────────────────────
DOC_COLORS = {
    "ordre_de_virement": "FFBDD7EE",
    "annexe_i":          "FFD9EAD3",
    "annexe_ii":         "FFE2D0ED",
    "bulletin_de_paie":  "FFFCE5CD",
    "meta":              "FFE2EFDA",
}
DOC_HEADER_COLORS = {
    "ordre_de_virement": "FF9DC3E6",
    "annexe_i":          "FFA9D18E",
    "annexe_ii":         "FFC499D1",
    "bulletin_de_paie":  "FFF4B183",
    "meta":              "FF70AD47",
}

print(f"✅ Config OK | Device: {DEVICE} | Batch: {GPU_BATCH_SIZE} | MaxTokens: {MAX_NEW_TOKENS}")

In [ ]:
FIELDS_OV = {
    "code_banque":          "Code banque (3 chiffres fixes, ex: 027)",
    "siege":                "Siège (5 chiffres, sous le mot Siège)",
    "racine":               "Racine (7 chiffres, sous le mot Racine)",
    "ordinal":              "Ordinal (3 chiffres, sous le mot Ordinal)",
    "cle":                  "Clé (2 chiffres, sous le mot Clé)",
    "compte_donneur":       "Numéro compte donneur complet (20 chiffres)",
    "cle_valide":           "Clé modulo 97 valide (True/False)",
    "monnaie":              "Monnaie (ex: DZD)",
    "montant_chiffres":     "Montant en chiffres",
    "montant_lettres":      "Montant en lettres",
    # Référence paiement décomposée en 4 champs
    "ref_mois":             "Mois de la référence paiement (ex: JANVIER)",
    "ref_annee":            "Année de la référence paiement (4 chiffres, ex: 2026)",
    "ref_tranche":          "Numéro de tranche/part (1 ou 2, null si absent)",
    "ref_complement":       "Complément (true si le mot complément est présent)",
    "devise":               "Devise (ex: DZD, EUR, USD)",
    "beneficiaire_nom":     "Bénéficiaire Nom/Prénom",
    "beneficiaire_iban":    "Bénéficiaire IBAN (sans espaces)",
    "beneficiaire_adresse": "Bénéficiaire Adresse complète",
    "swift":                "SWIFT/BIC banque bénéficiaire (8 ou 11 car.)",
    "banque_nom":           "Nom banque bénéficiaire",
}

FIELDS_ANN1 = {
    "nom":            "Nom du signataire",
    "date_signature": "Date de signature",
}

FIELDS_ANN2 = {
    "nom_prenom":           "Nom et prénom salarié",
    "salaire_mois":         "Mois du salaire (ex: JANVIER 2026)",
    "part_transferable":    "Part transférable (montant numérique)",
    "domiciliation":        "Domiciliation import (Oui/Non)",
    "numero_domiciliation": "Numéro domiciliation",
    "date_domiciliation":   "Date domiciliation",
    "agence_domiciliation": "Agence domiciliation",
    "compte_donneur":       "Numéro compte donneur",
    "iban_beneficiaire":    "IBAN bénéficiaire",
}

FIELDS_BP = {
    "nom_prenom":       "Nom et prénom salarié",
    "matricule":        "Matricule salarié",
    "mois_bulletin":    "Mois du bulletin (ex: JANVIER 2026)",
    "salaire_base":     "Salaire de base (R030)",
    "salaire_brut":     "Salaire brut total (TOTAL GAIN)",
    "retenue_ss":       "Retenue sécu. sociale (R510)",
    "retenue_irg":      "Retenue IRG (R660)",
    "retenue_mutuelle": "Retenue mutuelle (R515)",
    "total_retenues":   "Total retenues",
    "net_a_payer":      "Net à payer",
}

DOC_FIELDS = {
    "ordre_de_virement": FIELDS_OV,
    "annexe_i":          FIELDS_ANN1,
    "annexe_ii":         FIELDS_ANN2,
    "bulletin_de_paie":  FIELDS_BP,
}

META_FIELDS = [
    "dossier_id", "file_name", "processed_at",
    "pages_trouvees", "pages_manquantes",
    "tokens_in", "tokens_out", "tokens_total",
    "score_dossier", "alertes",
]

# Champs autorisés par document (anti-contamination)
ALLOWED_FIELDS = {doc: set(fields.keys()) for doc, fields in DOC_FIELDS.items()}

# Liste complète des colonnes Excel
ALL_COLUMNS = []
for mf in META_FIELDS:
    ALL_COLUMNS.append(("meta", mf, f"meta|{mf}"))
for doc_key, fields in DOC_FIELDS.items():
    for field_key in fields:
        ALL_COLUMNS.append((doc_key, field_key, f"{doc_key}|{field_key}"))

print(f"✅ Schéma OK | {len(ALL_COLUMNS)} colonnes Excel")
print(f"   META={len(META_FIELDS)} | OV={len(FIELDS_OV)} | ANN1={len(FIELDS_ANN1)} | ANN2={len(FIELDS_ANN2)} | BP={len(FIELDS_BP)}")

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def parse_json(text: str) -> dict:
    try:
        m = re.search(r"\{.*\}", text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        data = {}
        for line in text.splitlines():
            line = line.strip().rstrip(",")
            if ":" in line:
                try:
                    k, v = line.split(":", 1)
                    k = k.strip().strip('"')
                    v = v.strip().strip('"')
                    data[k] = None if v.lower() in ("null", "none", "") else v
                except Exception:
                    continue
        return data


def clean_str(v):
    if v is None: return None
    s = re.sub(r"\s+", " ", str(v).strip())
    return None if not s or s.lower() in ("null", "none", "n/a", "") else s


def clean_upper(v):
    s = clean_str(v); return s.upper() if s else None


def clean_account(v):
    s = clean_str(v)
    return re.sub(r"[^A-Za-z0-9]", "", s).upper() if s else None


def to_float(v):
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip().replace(" ", "").replace("\xa0", "")
    if s.count(",") == 1 and "." not in s: s = s.replace(",", ".")
    elif "." in s and "," in s: s = s.replace(",", "")
    elif s.count(",") > 1: s = s.replace(",", "")
    try: return float(s)
    except: return None


def validate_iban(v):
    s = clean_account(v)
    if not s or not re.match(r"^[A-Z]{2}[0-9A-Z]{13,34}$", s): return False
    try: return iban_lib.is_valid(s)
    except: return False


def resize_img(img: Image.Image, max_side: int = IMAGE_MAX_SIZE) -> Image.Image:
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w * r), int(h * r)), Image.LANCZOS)


def clean_pymupdf_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {3,}", "  ", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)
    return text.strip()


# ── Décomposition référence paiement ──────────────────────────────────────────
MOIS_MAP = {
    "JANVIER":  "JANVIER",  "JAN": "JANVIER",
    "FEVRIER":  "FEVRIER",  "FEV": "FEVRIER",  "FÉVRIER": "FEVRIER",  "FÉV": "FEVRIER",
    "MARS":     "MARS",
    "AVRIL":    "AVRIL",    "AVR": "AVRIL",
    "MAI":      "MAI",
    "JUIN":     "JUIN",
    "JUILLET":  "JUILLET",  "JUL": "JUILLET",
    "AOUT":     "AOUT",     "AOU": "AOUT",     "AOÛT": "AOUT",     "AÔU": "AOUT",
    "SEPTEMBRE":"SEPTEMBRE","SEP": "SEPTEMBRE",
    "OCTOBRE":  "OCTOBRE",  "OCT": "OCTOBRE",
    "NOVEMBRE": "NOVEMBRE", "NOV": "NOVEMBRE",
    "DECEMBRE": "DECEMBRE", "DEC": "DECEMBRE", "DÉCEMBRE": "DECEMBRE", "DÉC": "DECEMBRE",
}


def parse_reference_paiement(raw: str):
    """
    Décompose la référence paiement en 4 champs :
      (ref_mois, ref_annee, ref_tranche, ref_complement)

    Exemples :
      "JANVIER 26 Part 1"           -> JANVIER, 2026, 1, False
      "JANVIER 26 Part 1 complement"-> JANVIER, 2026, 1, True
      "JANVIER 26 Part 2"           -> JANVIER, 2026, 2, False
      "DECEMBRE.2025"               -> DECEMBRE, 2025, None, False
      "JANVIER 2026"                -> JANVIER, 2026, None, False
    """
    if not raw:
        return None, None, None, False

    s = str(raw).strip().upper()
    s = re.sub(r"[.\-/]", " ", s)
    s = re.sub(r"\s+", " ", s)

    has_complement = bool(re.search(r"COMPL[EÉ]MENT|COMPL\b", s))

    ref_mois = None
    for pat, normalized in MOIS_MAP.items():
        if re.search(r"\b" + pat + r"\b", s):
            ref_mois = normalized
            break

    ref_annee = None
    m_annee = re.search(r"\b(20\d{2})\b", s)
    if m_annee:
        ref_annee = int(m_annee.group(1))
    else:
        m_annee2 = re.search(r"\b(\d{2})\b", s)
        if m_annee2:
            ref_annee = 2000 + int(m_annee2.group(1))

    ref_tranche = None
    m_part = re.search(r"PART\s*(\d+)", s)
    if m_part:
        ref_tranche = int(m_part.group(1))
    else:
        m_tranche = re.search(r"TRANCHE\s*(\d+)", s)
        if m_tranche:
            ref_tranche = int(m_tranche.group(1))

    return ref_mois, ref_annee, ref_tranche, has_complement


# ── Tests de parse_reference_paiement ────────────────────────────────────────
_tests = [
    ("JANVIER 26 Part 1",              ("JANVIER", 2026, 1, False)),
    ("JANVIER 26 Part 1 complement",   ("JANVIER", 2026, 1, True)),
    ("JANVIER 26 Part 2",              ("JANVIER", 2026, 2, False)),
    ("DECEMBRE.2025",                  ("DECEMBRE", 2025, None, False)),
    ("JANVIER 2026",                   ("JANVIER", 2026, None, False)),
    ("JANVIER 26 Part 1 complément",   ("JANVIER", 2026, 1, True)),
]
ok = all(parse_reference_paiement(inp) == exp for inp, exp in _tests)
print(f"✅ Utilitaires OK")
print(f"   Tests parse_reference_paiement : {'✅ OK' if ok else '❌ ÉCHEC'}")
if not ok:
    for inp, exp in _tests:
        got = parse_reference_paiement(inp)
        if got != exp:
            print(f"   FAIL: {inp!r} -> {got} != {exp}")

In [ ]:
_OV_VIERGE_PIL = None  # cache global


def load_ov_vierge():
    """Charge l'image vierge OV une seule fois et la met en cache."""
    global _OV_VIERGE_PIL
    if _OV_VIERGE_PIL is not None:
        return _OV_VIERGE_PIL
    if OV_VIERGE_PATH.exists():
        img = Image.open(OV_VIERGE_PATH).convert("RGB")
        _OV_VIERGE_PIL = resize_img(img)
        print(f"✅ Image OV vierge chargée : {_OV_VIERGE_PIL.size}")
    else:
        print(f"⚠️  Image OV vierge introuvable : {OV_VIERGE_PATH}")
        print("   -> Fallback sur template texte ASCII")
    return _OV_VIERGE_PIL


load_ov_vierge()

In [ ]:
BASE_SYSTEM = (
    "Tu es un moteur OCR expert en documents bancaires algériens "
    "(transferts salaires à l'étranger, Instruction n°02-98). "
    "RÈGLES ABSOLUES : "
    "1) Extrais UNIQUEMENT les données du document fourni. "
    "2) Si un champ est absent ou illisible -> null. "
    "3) N'invente aucune valeur. "
    "4) Retourne UNIQUEMENT du JSON valide, sans explication, sans markdown."
)

TEMPLATE_OV_ASCII = (
    "FORMULAIRE OV BNP — STRUCTURE DE RÉFÉRENCE :\n"
    "  Agence              : ___________\n"
    "  Monnaie             : ___________    Montant chiffres : ___________\n"
    "  Montant en lettres  : ___________\n"
    "  Références paiement : ___________\n"
    "    ex: 'JANVIER 26 Part 1'  |  'JANVIER 26 Part 1 complément'  |  'JANVIER 26 Part 2'\n"
    "  Expéditeur (donneur d'ordre) :\n"
    "    Date    : ___________\n"
    "    Siège [5ch]   Racine [7ch]   Ordinal [3ch]   Clé [2ch]   Devise\n"
    "    _____         _______        ___             __           ___\n"
    "    <- ces chiffres sont SOUS les mots Siège/Racine/Ordinal/Clé\n"
    "  Bénéficiaire Nom/Prénom  : ___________\n"
    "  Bénéficiaire IBAN        : ___________\n"
    "  Bénéficiaire Adresse     : ___________\n"
    "  SWIFT                    : ___________\n"
    "  Banque bénéficiaire      : ___________\n"
)


def build_prompt_text(doc_type: str, text: str = "") -> str:
    """Construit le prompt texte pour un document natif (pymupdf)."""
    fields = DOC_FIELDS[doc_type]
    schema_json = json.dumps({k: None for k in fields}, ensure_ascii=False, indent=2)

    if doc_type == "ordre_de_virement":
        context = TEMPLATE_OV_ASCII
        hints = (
            "NUMÉRO DE COMPTE DONNEUR — Structure 20 chiffres :\n"
            "  Code banque(3) + Siège(5) + Racine(7) + Ordinal(3) + Clé(2) = 20 chiffres\n"
            "  Les chiffres se trouvent SOUS les mots Siège / Racine / Ordinal / Clé\n"
            "  Exemple : Siège=00731  Racine=0110942  Ordinal=001  Clé=79\n"
            "  -> code_banque=027, siege=00731, racine=0110942, ordinal=001, cle=79\n"
            "\n"
            "RÉFÉRENCE PAIEMENT — décomposer en 4 champs :\n"
            "  ref_mois      : mois en majuscules (ex: JANVIER, FEVRIER...)\n"
            "  ref_annee     : année sur 4 chiffres. Si 2 chiffres, préfixer 20 (ex: 26->2026)\n"
            "  ref_tranche   : numéro de Part/Tranche (1 ou 2), null si absent\n"
            "  ref_complement: true si le texte contient 'complément' ou 'complement', sinon false\n"
            "  Exemples :\n"
            "    'JANVIER 26 Part 1'            -> ref_mois=JANVIER, ref_annee=2026, ref_tranche=1, ref_complement=false\n"
            "    'JANVIER 26 Part 1 complément' -> ref_mois=JANVIER, ref_annee=2026, ref_tranche=1, ref_complement=true\n"
            "    'JANVIER 26 Part 2'            -> ref_mois=JANVIER, ref_annee=2026, ref_tranche=2, ref_complement=false\n"
            "    'DECEMBRE.2025'                -> ref_mois=DECEMBRE, ref_annee=2025, ref_tranche=null, ref_complement=false\n"
        )
    elif doc_type == "annexe_i":
        context = "Document : Annexe I (Instruction n°02-98 du 21 Mai 1998)\n"
        hints = ""
    elif doc_type == "annexe_ii":
        context = "Document : Annexe II (Instruction n°02-98 du 21 Mai 1998)\n"
        hints = (
            "- domiciliation : 'Oui' si mention DOMICILIATION IMPORT présente, sinon 'Non'\n"
            "- part_transferable : montant numérique uniquement\n"
        )
    elif doc_type == "bulletin_de_paie":
        context = "Document : Bulletin de Paie\n"
        hints = (
            "- salaire_base     : ligne R030 SALAIRE DE BASE colonne GAIN\n"
            "- salaire_brut     : ligne TOTAL colonne GAIN\n"
            "- retenue_ss       : ligne R510 RETENUE SECU.SLE colonne RETENUE\n"
            "- retenue_irg      : ligne R660 RETENUE IRG colonne RETENUE\n"
            "- retenue_mutuelle : ligne R515 RETENUE MUTUELLE colonne RETENUE\n"
            "- total_retenues   : ligne TOTAL colonne RETENUE\n"
            "- net_a_payer      : valeur NET A PAYER en bas du bulletin\n"
            "Montants : numérique sans symbole (ex: 519915.45)\n"
        )
    else:
        context = ""
        hints = ""

    return (
        f"{context}\n"
        f"TEXTE DU DOCUMENT :\n{text}\n\n"
        f"{hints}\n"
        f"Retourne UNIQUEMENT ce JSON :\n{schema_json}\n"
        "Règle : si un champ est absent -> null."
    )


def build_messages_ov_vision(page_image: Image.Image) -> list:
    """Mode vision OV : image vierge (référence) + image remplie (client)."""
    ov_vierge   = load_ov_vierge()
    fields      = DOC_FIELDS["ordre_de_virement"]
    schema_json = json.dumps({k: None for k in fields}, ensure_ascii=False, indent=2)

    if ov_vierge is not None:
        prompt_text = (
            "La PREMIÈRE image est le formulaire ORDRE DE VIREMENT BNP vierge (cases vides).\n"
            "La DEUXIÈME image est le même formulaire REMPLI par le client.\n"
            "\n"
            "En te basant sur la structure du formulaire vierge (image 1),\n"
            "extrais uniquement les cases remplies dans le document client (image 2).\n"
            "\n"
            "NUMÉRO DE COMPTE DONNEUR :\n"
            "Sur le formulaire, cherche la ligne de chiffres DIRECTEMENT\n"
            "EN DESSOUS des mots : Siège | Racine | Ordinal | Clé\n"
            "Ces mots sont des entêtes — les chiffres du compte sont juste en dessous.\n"
            "\n"
            "Structure du compte algérien BNP — 20 chiffres :\n"
            "  Code banque : 3 chiffres fixes (027)\n"
            "  Siège       : 5 chiffres (ex: 00731)\n"
            "  Racine      : 7 chiffres (ex: 0110942)\n"
            "  Ordinal     : 3 chiffres fixes (ex: 001)\n"
            "  Clé         : 2 chiffres (ex: 79)\n"
            "\n"
            "RÈGLE EXTRACTION COMPTE :\n"
            "1. Repérer les mots Siège / Racine / Ordinal / Clé sur le formulaire\n"
            "2. Lire les chiffres écrits JUSTE EN DESSOUS de chaque mot\n"
            "3. Ignorer tous les autres chiffres (montant, IBAN, etc.)\n"
            "\n"
            "Exemple :\n"
            "  Siège    Racine    Ordinal  Clé\n"
            "  00731    0110942   001      79\n"
            "-> code_banque=027, siege=00731, racine=0110942, ordinal=001, cle=79\n"
            "\n"
            "RÉFÉRENCE PAIEMENT — décomposer en 4 champs :\n"
            "  ref_mois      : mois en majuscules (JANVIER, FEVRIER...)\n"
            "  ref_annee     : année sur 4 chiffres (26->2026)\n"
            "  ref_tranche   : numéro de Part/Tranche (1 ou 2), null si absent\n"
            "  ref_complement: true si 'complément'/'complement' présent, sinon false\n"
            "  Exemples :\n"
            "    'JANVIER 26 Part 1'            -> ref_mois=JANVIER, ref_annee=2026, ref_tranche=1, ref_complement=false\n"
            "    'JANVIER 26 Part 1 complément' -> ref_mois=JANVIER, ref_annee=2026, ref_tranche=1, ref_complement=true\n"
            "    'JANVIER 26 Part 2'            -> ref_mois=JANVIER, ref_annee=2026, ref_tranche=2, ref_complement=false\n"
            "    'DECEMBRE.2025'                -> ref_mois=DECEMBRE, ref_annee=2025, ref_tranche=null, ref_complement=false\n"
            "\n"
            "AUTRES CHAMPS :\n"
            "- devise             : lettres uniquement (DZD, EUR, USD)\n"
            "- swift              : 8 ou 11 caractères sans espaces\n"
            "- beneficiaire_iban  : sans espaces, concaténé\n"
            "- compte_donneur     : 20 chiffres complet\n"
            "\n"
            f"Retourne UNIQUEMENT ce JSON :\n{schema_json}\n"
            "Règle : si une case est vide -> null."
        )
        return [
            {"role": "system", "content": [{"type": "text", "text": BASE_SYSTEM}]},
            {"role": "user", "content": [
                {"type": "image", "image": ov_vierge},
                {"type": "image", "image": page_image},
                {"type": "text",  "text": prompt_text},
            ]},
        ]
    else:
        # Fallback texte ASCII
        prompt_text = build_prompt_text("ordre_de_virement", "")
        return [
            {"role": "system", "content": [{"type": "text", "text": BASE_SYSTEM}]},
            {"role": "user", "content": [
                {"type": "image", "image": page_image},
                {"type": "text",  "text": prompt_text},
            ]},
        ]


def build_messages(page: dict, doc_type: str) -> list:
    """Construit les messages selon le type de document et le mode détecté."""
    if doc_type == "ordre_de_virement" and page["mode"] == "vision":
        return build_messages_ov_vision(page["image"])

    prompt_text = build_prompt_text(doc_type, page.get("text", ""))

    if page["mode"] == "vision":
        user_content = [
            {"type": "image", "image": page["image"]},
            {"type": "text",  "text": prompt_text},
        ]
    else:
        user_content = [{"type": "text", "text": prompt_text}]

    return [
        {"role": "system", "content": [{"type": "text", "text": BASE_SYSTEM}]},
        {"role": "user",   "content": user_content},
    ]


print("✅ Prompts OK")

In [ ]:
print(f"[MODEL] Chargement depuis : {MODEL_PATH}")
t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()
model.to(DEVICE)

print(f"✅ Modèle chargé en {time.time()-t0:.1f}s | Device: {DEVICE}")

In [ ]:
# Mapping fixe BNP : index page (0-based) -> type document
PAGE_TO_DOC = {
    0: "ordre_de_virement",
    1: "annexe_i",
    2: "annexe_ii",
    3: "bulletin_de_paie",
}


def load_pdf_pages(pdf_path: Path, zoom: float = 2.0) -> list:
    """Retourne liste de dicts : {page_num, image (PIL), text (str), mode (text|vision)}"""
    doc    = fitz.open(pdf_path)
    matrix = fitz.Matrix(zoom, zoom)
    pages  = []
    for i in range(len(doc)):
        page      = doc.load_page(i)
        raw_text  = page.get_text("text")
        clean_txt = clean_pymupdf_text(raw_text)
        pix = page.get_pixmap(matrix=matrix, alpha=False)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img = resize_img(img)
        mode = "text" if len(clean_txt) >= MIN_TEXT_CHARS else "vision"
        pages.append({
            "page_num": i + 1,
            "image":    img,
            "text":     clean_txt,
            "mode":     mode,
        })
    doc.close()
    return pages


def load_image_page(file_path: Path) -> list:
    img = Image.open(file_path).convert("RGB")
    return [{"page_num": 1, "image": resize_img(img), "text": "", "mode": "vision"}]


def load_file(file_path: Path) -> list:
    ext = file_path.suffix.lower()
    if ext == ".pdf":        return load_pdf_pages(file_path)
    if ext in SUPPORTED_EXT: return load_image_page(file_path)
    raise ValueError(f"Extension non supportée : {ext}")


print("✅ Chargement PDF OK — mode auto texte/vision")

In [ ]:
def run_single(page: dict, doc_type: str) -> dict:
    """Traitement d'une seule page isolée (OV vision = 2 images)."""
    messages = build_messages(page, doc_type)

    images = []
    for msg in messages:
        for block in (msg.get("content") or []):
            if isinstance(block, dict) and block.get("type") == "image":
                images.append(block["image"])

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text],
        images=images if images else None,
        return_tensors="pt",
    ).to(DEVICE)

    attn   = inputs.get("attention_mask")
    tok_in = int(attn.sum().item()) if attn is not None else inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            use_cache=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    eos_id    = processor.tokenizer.eos_token_id
    pos       = (generated == eos_id).nonzero(as_tuple=True)[0]
    tok_out   = int(pos[0].item()) if len(pos) > 0 else len(generated)

    raw = processor.decode(generated, skip_special_tokens=True,
                           clean_up_tokenization_spaces=True)
    return {"data": parse_json(raw), "tokens_in": tok_in, "tokens_out": tok_out}


def run_batch(pages_with_types: list) -> list:
    """
    pages_with_types : [(page_dict, doc_type), ...]
    Retourne         : [{data, tokens_in, tokens_out}, ...]

    Stratégie :
      - OV mode vision -> run_single (isolé, 2 images)
      - Tout le reste  -> batch groupé
    """
    results = [None] * len(pages_with_types)

    solo_indices  = []
    batch_indices = []
    for i, (page, doc_type) in enumerate(pages_with_types):
        if doc_type == "ordre_de_virement" and page["mode"] == "vision":
            solo_indices.append(i)
        else:
            batch_indices.append(i)

    # Traitement solo OV vision
    for i in solo_indices:
        page, doc_type = pages_with_types[i]
        results[i] = run_single(page, doc_type)

    # Traitement batch autres pages
    if batch_indices:
        batch_pages   = [pages_with_types[i] for i in batch_indices]
        messages_list = [build_messages(p, dt) for p, dt in batch_pages]

        all_images = []
        for msgs in messages_list:
            for msg in msgs:
                for block in (msg.get("content") or []):
                    if isinstance(block, dict) and block.get("type") == "image":
                        all_images.append(block["image"])

        texts = [
            processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            for msgs in messages_list
        ]
        inputs = processor(
            text=texts,
            images=all_images if all_images else None,
            padding=True,
            return_tensors="pt",
        ).to(DEVICE)

        attn        = inputs.get("attention_mask")
        tok_in_list = (attn.sum(dim=1).tolist() if attn is not None
                       else [inputs["input_ids"].shape[1]] * len(batch_pages))

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                temperature=None,
                top_p=None,
                use_cache=True,
                pad_token_id=processor.tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]
        generated  = output_ids[:, prompt_len:]
        eos_id     = processor.tokenizer.eos_token_id
        texts_out  = processor.batch_decode(
            generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )

        for j, (orig_i, text_out, tok_in) in enumerate(
            zip(batch_indices, texts_out, tok_in_list)
        ):
            row     = generated[j]
            pos     = (row == eos_id).nonzero(as_tuple=True)[0]
            tok_out = int(pos[0].item()) if len(pos) > 0 else len(row)
            results[orig_i] = {
                "data":       parse_json(text_out),
                "tokens_in":  int(tok_in),
                "tokens_out": tok_out,
            }

    return results


print("✅ Inférence GPU OK")
print("   OV vision  -> run_single (isolé, 2 images)")
print("   Autres     -> run_batch  (groupé)")

In [ ]:
def post_validate(raw_data: dict, doc_type: str) -> dict:
    """Garde uniquement les champs autorisés pour ce type de document."""
    allowed = ALLOWED_FIELDS.get(doc_type, set())
    cleaned = {k: v for k, v in raw_data.items() if k in allowed}
    for k in allowed:
        if k not in cleaned:
            cleaned[k] = None
    return cleaned


def reconstruct_compte_bnp(code_banque, siege, racine, ordinal, cle):
    """
    Reconstruit et valide le numéro de compte BNP — 20 chiffres :
      Code banque(3) + Siège(5) + Racine(7) + Ordinal(3) + Clé(2) = 20ch
    Valide la clé par modulo 97.
    """
    parts  = [str(x or "") for x in [code_banque, siege, racine, ordinal, cle]]
    digits = re.sub(r"\D", "", "".join(parts))

    if not digits:
        return None, None, None, None, None, None

    if len(digits) == 20:
        pass
    elif len(digits) == 19:
        digits = "0" + digits
    elif len(digits) > 20:
        digits = digits[:20]
    else:
        return code_banque, siege, racine, ordinal, cle, None

    cb = digits[0:3]    # Code banque (027)
    sg = digits[3:8]    # Siège       (00731)
    ra = digits[8:15]   # Racine      (0110942)
    od = digits[15:18]  # Ordinal     (001)
    cl = digits[18:20]  # Clé         (79)

    numero_sans_cle = cb + sg + ra + od
    cle_calculee    = str(97 - (int(numero_sans_cle) % 97)).zfill(2)
    cle_valide      = (cle_calculee == cl)

    if not cle_valide:
        for od_try in ["001", "000", od]:
            test = cb + sg + ra + od_try
            if str(97 - (int(test) % 97)).zfill(2) == cl:
                od = od_try
                cle_valide = True
                break

    compte_complet = cb + sg + ra + od + cl
    return cb, sg, ra, od, cl, compte_complet if cle_valide else None


def normalize_doc(data: dict, doc_type: str) -> dict:
    """Post-valide + normalise les types selon le document."""
    data = post_validate(data, doc_type)

    if doc_type == "ordre_de_virement":
        # Reconstruction compte BNP
        cb, sg, ra, od, cl, compte_ok = reconstruct_compte_bnp(
            data.get("code_banque"), data.get("siege"), data.get("racine"),
            data.get("ordinal"),     data.get("cle")
        )
        data["code_banque"]    = cb
        data["siege"]          = sg
        data["racine"]         = ra
        data["ordinal"]        = od
        data["cle"]            = cl
        data["compte_donneur"] = compte_ok or data.get("compte_donneur")
        data["cle_valide"]     = (compte_ok is not None)

        # Normalisation référence paiement :
        # Si le modèle n'a pas retourné ref_mois (champ absent ou vide),
        # on essaie de récupérer un champ brut "reference_paiement" et de le parser.
        if not data.get("ref_mois"):
            raw_ref = data.pop("reference_paiement", None)
            if raw_ref:
                m, a, t, c = parse_reference_paiement(str(raw_ref))
                data["ref_mois"]       = m
                data["ref_annee"]      = a
                data["ref_tranche"]    = t
                data["ref_complement"] = c

        # Normaliser les types des champs ref_* extraits par le modèle
        if data.get("ref_annee") is not None:
            try:
                y = int(str(data["ref_annee"]).strip())
                if y < 100: y += 2000
                data["ref_annee"] = y
            except Exception:
                data["ref_annee"] = None

        if data.get("ref_tranche") is not None:
            try:
                data["ref_tranche"] = int(str(data["ref_tranche"]).strip())
            except Exception:
                data["ref_tranche"] = None

        if data.get("ref_complement") is not None:
            v = data["ref_complement"]
            if isinstance(v, str):
                data["ref_complement"] = v.lower() in ("true", "oui", "1", "yes")
            else:
                data["ref_complement"] = bool(v)

    elif doc_type == "bulletin_de_paie":
        for field in ["salaire_base", "salaire_brut", "retenue_ss",
                      "retenue_irg", "retenue_mutuelle", "total_retenues", "net_a_payer"]:
            data[field] = to_float(data.get(field))

    elif doc_type == "annexe_ii":
        data["part_transferable"] = to_float(data.get("part_transferable"))
        if data.get("domiciliation"):
            v = str(data["domiciliation"]).strip().upper()
            data["domiciliation"] = "Oui" if v in ("OUI", "O", "YES", "TRUE", "1") else "Non"

    return data


print("✅ Post-validation & normalisation OK")

In [ ]:
def compute_controls(docs: dict) -> dict:
    """docs : {doc_type: data_dict} -> dict de contrôles + score + alertes"""
    ov   = docs.get("ordre_de_virement", {})
    ann2 = docs.get("annexe_ii",         {})
    bp   = docs.get("bulletin_de_paie",  {})

    montant_ov = to_float(ov.get("montant_chiffres"))
    part_trans = to_float(ann2.get("part_transferable"))
    net_bp     = to_float(bp.get("net_a_payer"))
    iban_ov    = clean_account(ov.get("beneficiaire_iban"))
    iban_a2    = clean_account(ann2.get("iban_beneficiaire"))
    cpt_ov     = clean_account(ov.get("compte_donneur"))
    cpt_a2     = clean_account(ann2.get("compte_donneur"))
    nom_a2     = clean_upper(ann2.get("nom_prenom"))
    nom_bp     = clean_upper(bp.get("nom_prenom"))

    ctrl = {}
    ctrl["ctrl_montant_ov_le_part"] = (
        None if montant_ov is None or part_trans is None else (montant_ov <= part_trans))
    ctrl["ctrl_part_le_95pct_net"] = (
        None if part_trans is None or net_bp is None else (part_trans <= net_bp * 0.95))
    ctrl["ctrl_iban_coherent"] = (
        None if not iban_ov or not iban_a2 else (iban_ov == iban_a2))
    ctrl["ctrl_compte_coherent"] = (
        None if not cpt_ov or not cpt_a2 else (cpt_ov == cpt_a2))
    ctrl["ctrl_nom_coherent"] = (
        None if not nom_a2 or not nom_bp else (nom_a2 == nom_bp))
    ctrl["ctrl_iban_valide"] = validate_iban(iban_ov or iban_a2)
    ctrl["ctrl_cle_valide"]  = bool(ov.get("cle_valide"))
    ctrl["taux_transfert"]   = (
        None if part_trans is None or not net_bp else round(part_trans / net_bp, 4))

    valid = [v for v in ctrl.values() if isinstance(v, bool)]
    ctrl["score_dossier"] = round(sum(valid) / len(valid) * 100, 1) if valid else None

    alertes = []
    if ctrl["ctrl_montant_ov_le_part"] is False: alertes.append("montant_ov > part_transferable")
    if ctrl["ctrl_part_le_95pct_net"]  is False: alertes.append("part_transferable > 95% net")
    if ctrl["ctrl_iban_coherent"]      is False: alertes.append("IBAN incohérent OV/ANN2")
    if ctrl["ctrl_compte_coherent"]    is False: alertes.append("compte_donneur incohérent")
    if ctrl["ctrl_nom_coherent"]       is False: alertes.append("nom incohérent ANN2/BP")
    if not ctrl["ctrl_iban_valide"]:             alertes.append("IBAN invalide")
    if not ctrl["ctrl_cle_valide"]:              alertes.append("clé compte invalide")
    ctrl["alertes"] = "; ".join(alertes) if alertes else None

    return ctrl


print("✅ Contrôles métier OK")

In [ ]:
def create_excel(path: Path) -> Workbook:
    wb = Workbook()
    ws = wb.active
    ws.title = "Dossiers"

    # Ligne 1 : groupes fusionnés colorés
    col_idx   = 1
    group_map = defaultdict(list)
    for doc_key, field_key, col_label in ALL_COLUMNS:
        group_map[doc_key].append(col_idx)
        col_idx += 1

    for doc_key, cols in group_map.items():
        start_col, end_col = cols[0], cols[-1]
        if start_col < end_col:
            ws.merge_cells(
                start_row=1, start_column=start_col,
                end_row=1,   end_column=end_col
            )
        cell           = ws.cell(row=1, column=start_col)
        cell.value     = doc_key.upper().replace("_", " ")
        cell.font      = Font(bold=True, color="FFFFFFFF", name="Arial", size=11)
        cell.fill      = PatternFill("solid", start_color=DOC_HEADER_COLORS.get(doc_key, "FF808080"))
        cell.alignment = Alignment(horizontal="center", vertical="center")
        ws.row_dimensions[1].height = 22

    # Ligne 2 : noms de colonnes
    for i, (doc_key, field_key, col_label) in enumerate(ALL_COLUMNS, start=1):
        cell           = ws.cell(row=2, column=i)
        cell.value     = col_label
        cell.font      = Font(bold=True, name="Arial", size=9)
        cell.fill      = PatternFill("solid", start_color=DOC_COLORS.get(doc_key, "FFF2F2F2"))
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = Border(bottom=Side(style="thin"), right=Side(style="hair"))
        ws.column_dimensions[get_column_letter(i)].width = 22
    ws.row_dimensions[2].height = 35

    n_meta = len(META_FIELDS)
    ws.freeze_panes = ws.cell(row=3, column=n_meta + 1)

    wb.save(path)
    return wb


class ExcelWriter:
    """Buffer en RAM, flush toutes les EXCEL_FLUSH_EVERY lignes."""

    def __init__(self, path: Path):
        self.path    = path
        self.wb      = create_excel(path) if not path.exists() else load_workbook(path)
        self._buffer = []
        self._count  = 0
        self._doc_fills = {
            doc_key: PatternFill("solid", start_color=color)
            for doc_key, color in DOC_COLORS.items()
        }

    def add_row(self, row_data: dict):
        self._buffer.append(row_data)
        self._count += 1
        if self._count % EXCEL_FLUSH_EVERY == 0:
            self.flush()

    def flush(self):
        if not self._buffer:
            return
        ws = self.wb["Dossiers"]
        next_row = ws.max_row + 1
        for row_data in self._buffer:
            for col_idx, (doc_key, field_key, col_label) in enumerate(ALL_COLUMNS, start=1):
                if doc_key == "meta":
                    val = row_data.get(field_key)
                else:
                    doc_data = row_data.get(doc_key, {}) or {}
                    val      = doc_data.get(field_key)
                cell       = ws.cell(row=next_row, column=col_idx)
                cell.value = val
                fill       = self._doc_fills.get(doc_key)
                if fill:
                    cell.fill = fill
            next_row += 1
        self._buffer.clear()
        self.wb.save(self.path)

    def close(self):
        self.flush()
        self.wb.save(self.path)


print("✅ ExcelWriter OK")

In [ ]:
def load_checkpoint() -> dict:
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_checkpoint(cp: dict):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(cp, f, ensure_ascii=False, indent=2)


checkpoint = load_checkpoint()
print(f"✅ Checkpoint : {len(checkpoint)} dossier(s) déjà traité(s)")

In [ ]:
def process_file(file_path: Path, xl: ExcelWriter, cp: dict) -> dict:
    file_name  = file_path.name
    dossier_id = file_path.stem
    file_hash  = sha256_file(file_path)

    if file_hash in cp:
        return {"status": "skipped", "dossier_id": dossier_id}

    processed_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    pages        = load_file(file_path)

    pages_typed = []
    for page in pages:
        idx      = page["page_num"] - 1
        doc_type = PAGE_TO_DOC.get(idx)
        if doc_type:
            pages_typed.append((page, doc_type))

    extracted_docs = {}
    total_tok_in   = 0
    total_tok_out  = 0

    for batch_start in range(0, len(pages_typed), GPU_BATCH_SIZE):
        batch   = pages_typed[batch_start : batch_start + GPU_BATCH_SIZE]
        results = run_batch(batch)
        for (page, doc_type), res in zip(batch, results):
            total_tok_in  += res["tokens_in"]
            total_tok_out += res["tokens_out"]
            extracted_docs[doc_type] = normalize_doc(res["data"], doc_type)

    # Purge VRAM
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    all_docs     = set(PAGE_TO_DOC.values())
    found_docs   = set(extracted_docs.keys())
    missing_docs = sorted(all_docs - found_docs)
    controls     = compute_controls(extracted_docs)

    row_data = {
        "dossier_id":      dossier_id,
        "file_name":       file_name,
        "processed_at":    processed_at,
        "pages_trouvees":  ", ".join(sorted(found_docs)),
        "pages_manquantes": ", ".join(missing_docs) if missing_docs else None,
        "tokens_in":       total_tok_in,
        "tokens_out":      total_tok_out,
        "tokens_total":    total_tok_in + total_tok_out,
        "score_dossier":   controls.get("score_dossier"),
        "alertes":         controls.get("alertes"),
        **extracted_docs,
    }
    xl.add_row(row_data)

    # Archive JSON
    with open(JSON_ARCHIVE_DIR / f"{dossier_id}.json", "w", encoding="utf-8") as f:
        json.dump({
            "dossier_id":   dossier_id,
            "file_name":    file_name,
            "processed_at": processed_at,
            "docs":         extracted_docs,
            "controls":     controls,
        }, f, ensure_ascii=False, indent=2)

    cp[file_hash] = {"dossier_id": dossier_id, "processed_at": processed_at}

    return {
        "status":     "ok",
        "dossier_id": dossier_id,
        "tokens_in":  total_tok_in,
        "tokens_out": total_tok_out,
        "score":      controls.get("score_dossier"),
        "alertes":    controls.get("alertes"),
    }


print("✅ process_file OK")

In [ ]:
def run_pipeline(input_dir: Path = INPUT_DIR):
    files = sorted([
        p for p in input_dir.iterdir()
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXT
    ])
    print(f"📁 {len(files)} fichier(s) trouvé(s) dans {input_dir}")

    cp    = load_checkpoint()
    xl    = ExcelWriter(EXCEL_OUTPUT)
    stats = defaultdict(int)
    grand_tok_in = grand_tok_out = 0
    t0    = time.time()

    with tqdm(files, desc="Dossiers", unit="dos") as pbar:
        for file_path in pbar:
            try:
                res = process_file(file_path, xl, cp)
                stats[res["status"]] += 1

                if res["status"] == "ok":
                    grand_tok_in  += res.get("tokens_in",  0)
                    grand_tok_out += res.get("tokens_out", 0)

                if (stats["ok"] + stats["error"]) % 10 == 0:
                    save_checkpoint(cp)

                pbar.set_postfix({
                    "ok":   stats["ok"],
                    "skip": stats["skipped"],
                    "err":  stats["error"],
                    "tok↑": f"{grand_tok_in/1e6:.1f}M",
                    "tok↓": f"{grand_tok_out/1e6:.1f}M",
                })

            except Exception as e:
                stats["error"] += 1
                tqdm.write(f"  ❌ {file_path.name} : {e}")

    xl.close()
    save_checkpoint(cp)

    elapsed = time.time() - t0
    n = stats["ok"] or 1
    print(f"\n{'='*60}")
    print(f"  ✅ TERMINÉ en {elapsed/60:.1f} min")
    print(f"  Traités        : {stats['ok']}")
    print(f"  Skippés        : {stats['skipped']}")
    print(f"  Erreurs        : {stats['error']}")
    print(f"  Vitesse        : {n/elapsed:.2f} dos/s | {elapsed/n:.1f}s/dos")
    print(f"  Tokens IN      : {grand_tok_in:>12,} ({grand_tok_in/1e6:.2f}M)")
    print(f"  Tokens OUT     : {grand_tok_out:>12,} ({grand_tok_out/1e6:.2f}M)")
    print(f"  Excel          : {EXCEL_OUTPUT}")
    print(f"{'='*60}")


print("✅ run_pipeline prêt")

In [ ]:
# Décommenter et modifier TEST_FILE pour tester un dossier
TEST_FILE = None
# TEST_FILE = INPUT_DIR / "dossier_001.pdf"

if TEST_FILE and Path(TEST_FILE).exists():
    cp_test = load_checkpoint()
    xl_test = ExcelWriter(OUTPUT_DIR / "_test_single.xlsx")
    t0 = time.time()
    res = process_file(Path(TEST_FILE), xl_test, cp_test)
    xl_test.close()
    print(f"⏱  {time.time()-t0:.1f}s")
    print(json.dumps(res, ensure_ascii=False, indent=2, default=str))
else:
    print("Définir TEST_FILE pour tester un seul dossier.")

In [ ]:
TEST_DIR = Path("/mnt/data/transferts_in")   # <- modifier si besoin

pdfs = sorted(TEST_DIR.glob("*.pdf"))
print(f"📁 Répertoire : {TEST_DIR}")
print(f"📄 {len(pdfs)} PDF(s) trouvé(s)\n")

if not pdfs:
    print("⚠️  Aucun PDF trouvé. Vérifier le chemin TEST_DIR.")
else:
    cp  = load_checkpoint()
    xl  = ExcelWriter(OUTPUT_DIR / "_test_repertoire.xlsx")
    t0  = time.time()
    ok_count  = 0
    err_count = 0
    scores    = []

    for i, pdf in enumerate(pdfs, 1):
        sha = sha256_file(pdf)
        if sha in cp:
            print(f"[{i:>3}/{len(pdfs)}] ⏭️  SKIP : {pdf.name}")
            continue

        print(f"[{i:>3}/{len(pdfs)}] ⏳ {pdf.name}", end=" ... ", flush=True)
        t1 = time.time()
        try:
            res     = process_file(pdf, xl, cp)
            elapsed_1 = time.time() - t1
            score   = res.get("score") or 0
            alertes = res.get("alertes") or "aucune"
            scores.append(score)
            ok_count += 1
            print(f"✅ {elapsed_1:.1f}s | score={score}% | alertes={alertes}")
        except Exception as e:
            err_count += 1
            print(f"❌ ERREUR : {e}")

    xl.close()
    elapsed_total = time.time() - t0
    print(f"\n{'='*60}")
    print(f"⏱  Temps total    : {elapsed_total:.1f}s")
    print(f"   Temps moyen    : {elapsed_total/max(1,ok_count+err_count):.1f}s/dossier")
    print(f"✅ Succès         : {ok_count}/{len(pdfs)}")
    print(f"❌ Erreurs        : {err_count}/{len(pdfs)}")
    if scores:
        print(f"📊 Score moyen    : {sum(scores)/len(scores):.1f}%")
        print(f"   Score min/max  : {min(scores)}% / {max(scores)}%")
    print(f"\n📁 Excel          : {OUTPUT_DIR / '_test_repertoire.xlsx'}")

In [ ]:
# Décommenter pour lancer le batch complet
# run_pipeline()

In [ ]:
import pandas as pd

if EXCEL_OUTPUT.exists():
    df = pd.read_excel(EXCEL_OUTPUT, sheet_name="Dossiers", header=1)
    print(f"📊 {len(df)} dossiers")
    if "meta|score_dossier" in df.columns:
        print(f"Score moyen : {df['meta|score_dossier'].mean():.1f}%")
    print()
    print("── Tokens ───────────────────────────────")
    for col in ["meta|tokens_in", "meta|tokens_out", "meta|tokens_total"]:
        if col in df.columns:
            print(f"  {col.split('|')[1]:12s}: {df[col].sum():>12,.0f}")
    print()
    print("── Références paiement ──────────────────")
    for col in ["ordre_de_virement|ref_mois", "ordre_de_virement|ref_tranche",
                "ordre_de_virement|ref_complement"]:
        if col in df.columns:
            print(f"\n  {col}:")
            print(df[col].value_counts().head(6).to_string())
    print()
    print("── Alertes fréquentes ───────────────────")
    if "meta|alertes" in df.columns:
        a = df["meta|alertes"].dropna().str.split(";").explode().str.strip()
        print(a.value_counts().head(10).to_string())
    print()
    display(df.head(5))
else:
    print("Excel non encore généré.")